# 01 — Spark KPI + ML dataset
Lecture Mongo via pymongo → Spark local → écriture Mongo.


In [4]:
from src.velib_spark_local import get_spark, read_mongo_as_spark, build_kpi_station, build_ml_dataset, write_spark_to_mongo
spark = get_spark('velib-kpi-ml')
availability = read_mongo_as_spark(spark, 'availability_raw')
print('rows:', availability.count())
availability.printSchema()


[Stage 0:>                                                          (0 + 8) / 8]

rows: 1507
root
 |-- _id: string (nullable = true)
 |-- ts: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- station_id: long (nullable = true)
 |-- num_bikes_available: long (nullable = true)
 |-- num_docks_available: long (nullable = true)
 |-- is_installed: long (nullable = true)
 |-- is_renting: long (nullable = true)
 |-- is_returning: long (nullable = true)
 |-- last_reported: long (nullable = true)



In [5]:
availability.show(5, truncate=False)


+------------------------+--------------------------+--------------------------------+-----------+-------------------+-------------------+------------+----------+------------+-------------+
|_id                     |ts                        |timestamp                       |station_id |num_bikes_available|num_docks_available|is_installed|is_renting|is_returning|last_reported|
+------------------------+--------------------------+--------------------------------+-----------+-------------------+-------------------+------------+----------+------------+-------------+
|697f27abd38e6530f13111e8|2026-02-01T10:15:07.218000|2026-02-01T10:15:07.218215+00:00|213688169  |10                 |24                 |1           |1         |1           |1769939035   |
|697f27abd38e6530f13111e9|2026-02-01T10:15:07.218000|2026-02-01T10:15:07.218215+00:00|19179944124|16                 |11                 |1           |1         |1           |1769938991   |
|697f27abd38e6530f13111ea|2026-02-01T10:15:07.2180

In [6]:
kpi = build_kpi_station(availability)
kpi.show(10, truncate=False)
write_spark_to_mongo(kpi, 'kpi_station', mode='overwrite')
print('✅ écrit: velib.kpi_station')


CodeCache: size=131072Kb used=21138Kb max_used=21138Kb free=109933Kb
 bounds [0x00000001071f8000, 0x00000001086b8000, 0x000000010f1f8000]
 total_blobs=8534 nmethods=7629 adapters=816
 compilation: disabled (not enough contiguous free space left)


Java HotSpot(TM) 64-Bit Server VM warning: CodeCache is full. Compiler has been disabled.
Java HotSpot(TM) 64-Bit Server VM warning: Try increasing the code cache size using -XX:ReservedCodeCacheSize=


+-----------+---------------------+---------+---------+
|station_id |avg_availability_rate|avg_bikes|avg_docks|
+-----------+---------------------+---------+---------+
|14813797734|0.4444444444444444   |12.0     |15.0     |
|457813746  |0.4                  |6.0      |9.0      |
|66491397   |0.9259259259259259   |50.0     |4.0      |
|209063434  |0.5862068965517241   |17.0     |12.0     |
|128920403  |0.7037037037037037   |19.0     |8.0      |
|39296638   |0.08333333333333333  |4.0      |44.0     |
|19331981332|0.23809523809523808  |5.0      |16.0     |
|3906335682 |0.13043478260869565  |3.0      |20.0     |
|88230992   |0.3103448275862069   |9.0      |20.0     |
|129087587  |0.025                |1.0      |39.0     |
+-----------+---------------------+---------+---------+
only showing top 10 rows

✅ écrit: velib.kpi_station


In [7]:
ml = build_ml_dataset(availability)
ml.show(10, truncate=False)
write_spark_to_mongo(ml, 'ml_dataset', mode='overwrite')
print('✅ écrit: velib.ml_dataset')


+-----------+--------------------------------+----+---+-------------------+-------------------+
|station_id |timestamp                       |hour|dow|num_bikes_available|num_docks_available|
+-----------+--------------------------------+----+---+-------------------+-------------------+
|213688169  |2026-02-01T10:15:07.218215+00:00|11  |1  |10                 |24                 |
|19179944124|2026-02-01T10:15:07.218215+00:00|11  |1  |16                 |11                 |
|18795078746|2026-02-01T10:15:07.218215+00:00|11  |1  |8                  |19                 |
|36255      |2026-02-01T10:15:07.218215+00:00|11  |1  |0                  |19                 |
|251039991  |2026-02-01T10:15:07.218215+00:00|11  |1  |4                  |20                 |
|85002689   |2026-02-01T10:15:07.218215+00:00|11  |1  |16                 |43                 |
|2515829865 |2026-02-01T10:15:07.218215+00:00|11  |1  |21                 |1                  |
|516709288  |2026-02-01T10:15:07.218215+

In [ ]:
spark.stop()
